# Day 57 · Exercise 1: Config from Environment

**What you'll build:** Implement `load_config(env_vars, defaults)` — the 12-factor app pattern for reading all configuration from the environment. Every deployed app needs this: local defaults + production overrides.

## Setup (provided)

In [ ]:
import os


## Your Implementation

In [ ]:
def load_config(env_vars: dict, defaults: dict) -> dict:
    """Merge env_vars over defaults to produce a configuration dict.

    Args:
        env_vars: Dictionary of environment variable values (e.g. os.environ).
        defaults: Dictionary of default values for each config key.
    Returns:
        A dict where every key from defaults is present. If env_vars contains
        the same key, the env_vars value takes priority over the default.

    Example:
        defaults = {"MODEL": "llama3.2", "PORT": "8000", "DEBUG": "false"}
        env_vars = {"PORT": "9000"}
        load_config(env_vars, defaults)
        # → {"MODEL": "llama3.2", "PORT": "9000", "DEBUG": "false"}
    """
    # TODO:
    # 1. Start with a copy of defaults
    # 2. For each key in env_vars that is also in defaults, override the default
    # 3. Return the merged dict
    raise NotImplementedError


In [ ]:
def load_config(env_vars: dict, defaults: dict) -> dict:
    config = dict(defaults)
    for key, value in env_vars.items():
        if key in config:
            config[key] = value
    return config


## Check Your Work

In [ ]:
def _run_checks():
    score = 0
    total = 5

    def _chk(n, ok, msg):
        nonlocal score
        print(f"  {'✅' if ok else '❌'} Check {n}: {msg}")
        if ok:
            score += 1

    defaults = {"MODEL": "llama3.2", "PORT": "8000", "DEBUG": "false"}

    try:
        cfg = load_config({}, defaults)
    except NotImplementedError:
        for i in range(1, total + 1):
            print(f"  ❌ Check {i}: load_config not implemented")
        print(f"\nScore: 0 / {total}")
        return

    _chk(1, cfg == defaults,
         f"empty env_vars → config equals defaults (got {cfg})")

    cfg2 = load_config({"PORT": "9000"}, defaults)
    _chk(2, cfg2["PORT"] == "9000" and cfg2["MODEL"] == "llama3.2",
         f"PORT overridden, MODEL unchanged (got {cfg2})")

    cfg3 = load_config({"MODEL": "mistral", "PORT": "8080"}, defaults)
    _chk(3, cfg3["MODEL"] == "mistral" and cfg3["PORT"] == "8080",
         f"two overrides applied (got {cfg3})")

    # unknown env vars should not appear in the result
    cfg4 = load_config({"UNKNOWN_VAR": "x", "PORT": "7000"}, defaults)
    _chk(4, "UNKNOWN_VAR" not in cfg4 and cfg4["PORT"] == "7000",
         f"unknown env vars not injected (got {cfg4})")

    # defaults should not be mutated
    cfg5 = load_config({"PORT": "5000"}, defaults)
    _chk(5, defaults["PORT"] == "8000",
         f"defaults dict not mutated by load_config (PORT still {defaults['PORT']})")

    print(f"\nScore: {score} / {total}")
    if score == total:
        print("🎉 Exercise complete!")

_run_checks()


## Bonus Challenge

Extend `load_config` to support type coercion: add a `types` dict parameter mapping key names to callables (e.g. `{'PORT': int, 'DEBUG': bool}`). For each key in types, call `types[key](config[key])` to convert the string env var to the right Python type. Handle `bool` specially: 'true'/'1'/'yes' → True, anything else → False.

## Solution

<details>
<summary>Show solution</summary>

```python
def load_config(env_vars: dict, defaults: dict) -> dict:
    config = dict(defaults)
    for key, value in env_vars.items():
        if key in config:
            config[key] = value
    return config
```

**Why this works:** Start with `dict(defaults)` — a shallow copy — so the
caller's defaults dict is never mutated. Then iterate over `env_vars` and
override only keys that exist in the defaults. Unknown env vars (like `PATH`
or `HOME`) are silently ignored — you only want the config keys you declared.
This is the 12-factor app pattern: all config comes from the environment,
but the defaults give you a working local setup without any setup.

</details>